In [1]:
# =============================================================================
# 🥈 CAPA SILVER - TRANSFORMACIONES Y DELTA LAKE (RUTA CORREGIDA)
# Pipeline: GetTalent TP6 - Microsoft Fabric
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from notebookutils import mssparkutils
import requests
from datetime import datetime

# ⚙️ CONFIGURACIÓN SPARK
spark = SparkSession.builder \
    .appName("Pipeline_GetTalent_Silver_Fabric") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
    .config("spark.databricks.delta.properties.defaults.autoOptimize.optimizeWrite", "true") \
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false") \
    .getOrCreate()

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY")

# 📂 RUTAS CORREGIDAS
BRONZE_PATH = "abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/e9dcb661-05e8-46bd-be1f-a9cb5ee27446/Files/bronze/shop_carrito"

# ✅ CAMBIO CRÍTICO: Files en lugar de Tables
SILVER_BASE = "abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/4d897e0a-b68c-4840-9910-752862308e5b/Files/silver/shop_carrito"

print("✅ Configuración Silver inicializada")
print(f"📂 Origen Bronze: {BRONZE_PATH}")
print(f"💎 Destino Silver: {SILVER_BASE}")
print(f"💾 Formato: Delta Lake")
print(f"\n📋 7 datasets | Mínimo 2 transformaciones cada uno")
print(f"🌐 2 datasets con enriquecimiento via API\n")


# =============================================================================
# 🛒 DATASET 1/7: SHOP_CARRITO (CON API)
# =============================================================================

print("="*80)
print("🛒 DATASET 1/7: SHOP_CARRITO  (CON API)")
print("="*80)

df_carrito_bronze = spark.read.parquet(f"{BRONZE_PATH}/Shop_carrito")
print(f"📥 Registros en Bronze: {df_carrito_bronze.count():,}")

print("\n🔧 TRANSFORMACIÓN 1: Limpieza y normalización de tipos")

df_carrito_t1 = df_carrito_bronze \
    .filter(col("id").isNotNull()) \
    .dropDuplicates(["id"]) \
    .withColumn("fecha",       to_timestamp(col("fecha"), "M/d/yyyy HH:mm")) \
    .withColumn("monto",       col("monto").cast("decimal(12,2)")) \
    .withColumn("estado",      col("estado").cast("integer")) \
    .withColumn("estado_pago", col("estado_pago").cast("integer")) \
    .withColumn("id_cliente",  col("id_cliente").cast("integer")) \
    .withColumn("id_fpago",    col("id_fpago").cast("integer")) \
    .withColumn("id_campania", col("id_campania").cast("integer")) \
    .withColumn("localidad",   upper(trim(coalesce(col("localidad"), lit("SIN_DATOS"))))) \
    .withColumn("provincia",   upper(trim(coalesce(col("provincia"), lit("SIN_DATOS"))))) \
    .filter(col("monto") >= 0)

print(f"   ✅ Duplicados eliminados | Fechas parseadas | Tipos casteados")
print(f"   📊 Registros después de T1: {df_carrito_t1.count():,}")

print("\n🔧 TRANSFORMACIÓN 2: Columnas derivadas y segmentación")

df_carrito_t2 = df_carrito_t1 \
    .withColumn("anio",         year(col("fecha"))) \
    .withColumn("mes",          month(col("fecha"))) \
    .withColumn("dia",          dayofmonth(col("fecha"))) \
    .withColumn("dia_semana",   dayofweek(col("fecha"))) \
    .withColumn("hora",         hour(col("fecha"))) \
    .withColumn("segmento_monto",
        when(col("monto") < 5000,  "BAJO")
        .when((col("monto") >= 5000)  & (col("monto") < 20000), "MEDIO")
        .when((col("monto") >= 20000) & (col("monto") < 50000), "ALTO")
        .when(col("monto") >= 50000, "PREMIUM")
        .otherwise("NO_CLASIFICADO")
    )

print(f"   ✅ Columnas temporales + Segmentación aplicada")
print(f"   📊 Registros después de T2: {df_carrito_t2.count():,}")

print("\n🌐 TRANSFORMACIÓN 3 (CON API): Conversión de moneda")

def obtener_tipo_cambio():
    try:
        response = requests.get("https://api.exchangerate-api.com/v4/latest/USD", timeout=5)
        if response.status_code == 200:
            data  = response.json()
            rates = data.get("rates", {})
            return {
                "ars": rates.get("ARS", 1000.0),
                "eur": rates.get("EUR", 0.92),
                "brl": rates.get("BRL", 5.0),
                "fecha_api": data.get("date", datetime.now().strftime("%Y-%m-%d"))
            }
    except:
        pass
    return {"ars": 1000.0, "eur": 0.92, "brl": 5.0, "fecha_api": "2026-02-17"}

tc = obtener_tipo_cambio()
print(f"   ✅ API consultada: {tc['fecha_api']}")
print(f"   💱 1 USD = {tc['ars']} ARS | {tc['eur']} EUR | {tc['brl']} BRL")

df_carrito_final = df_carrito_t2 \
    .withColumn("monto_usd",         round(col("monto") / 1000, 2)) \
    .withColumn("monto_eur",         round((col("monto") / 1000) * lit(tc["eur"]), 2)) \
    .withColumn("monto_brl",         round((col("monto") / 1000) * lit(tc["brl"]), 2)) \
    .withColumn("fecha_tipo_cambio", lit(tc["fecha_api"])) \
    .withColumn("silver_timestamp",  current_timestamp())

print(f"   ✅ Columnas API agregadas")
print(f"   📊 Registros finales: {df_carrito_final.count():,}")

print("\n💾 Guardando: shop_carrito (Delta)")
tabla_path = f"{SILVER_BASE}/shop_carrito"
df_carrito_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(tabla_path)
print(f"✅ Guardado en: {tabla_path}\n")


# =============================================================================
# 👤 DATASET 2/7: SHOP_CLIENTES (CON API)
# =============================================================================

print("="*80)
print("👤 DATASET 2/7: SHOP_CLIENTES  (CON API)")
print("="*80)

df_clientes_bronze = spark.read.parquet(f"{BRONZE_PATH}/Shop_clientes")
print(f"📥 Registros en Bronze: {df_clientes_bronze.count():,}")

print("\n🔧 TRANSFORMACIÓN 1: Normalización")

df_clientes_t1 = df_clientes_bronze \
    .filter(col("id").isNotNull()) \
    .dropDuplicates(["id"]) \
    .withColumn("nombre_completo", concat_ws(" ", trim(col("nombre")), trim(col("apellido")))) \
    .withColumn("email_limpio",    lower(trim(coalesce(col("email"), lit("sin_email@domain.com"))))) \
    .withColumn("telefono_limpio", regexp_replace(col("telefono_particular"), "[^0-9+]", "")) \
    .withColumn("localidad",       upper(trim(coalesce(col("localidad"), lit("SIN_DATOS"))))) \
    .withColumn("provincia",       upper(trim(coalesce(col("provincia"),  lit("SIN_DATOS")))))

print(f"   ✅ Normalización aplicada")
print(f"   📊 Registros después de T1: {df_clientes_t1.count():,}")

print("\n🔧 TRANSFORMACIÓN 2: Segmentación")

df_clientes_t2 = df_clientes_t1 \
    .withColumn("segmento_provincia",
        when(col("provincia").isin(["SANTIAGO", "METROPOLITANA"]), "CENTRO")
        .when(col("provincia").isin(["VALPARAISO", "CONCEPCION"]), "URBANO")
        .otherwise("REGIONAL")
    ) \
    .withColumn("sexo_desc",
        when(col("sexo") == "M", "MASCULINO")
        .when(col("sexo") == "F", "FEMENINO")
        .otherwise("NO_ESPECIFICADO")
    )

print(f"   ✅ Segmentación aplicada")
print(f"   📊 Registros después de T2: {df_clientes_t2.count():,}")

print("\n🌐 TRANSFORMACIÓN 3 (CON API): REST Countries")

def obtener_info_pais(id_pais):
    paises = {1: "Argentina", 2: "Brazil", 3: "Chile"}
    nombre = paises.get(id_pais, "Chile")
    try:
        response = requests.get(f"https://restcountries.com/v3.1/name/{nombre}", timeout=5)
        if response.status_code == 200:
            data = response.json()[0]
            moneda = list(data.get("currencies", {}).keys())
            return {
                "pais_nombre": data.get("name", {}).get("common", nombre),
                "region":      data.get("region", "Americas"),
                "capital":     data.get("capital", ["Desconocida"])[0] if data.get("capital") else "Desconocida",
                "poblacion":   data.get("population", 0),
                "moneda":      moneda[0] if moneda else "USD"
            }
    except:
        pass
    return {"pais_nombre": nombre, "region": "Americas", "capital": "Desconocida", "poblacion": 0, "moneda": "USD"}

paises_unicos = [r.id_pais for r in df_clientes_t2.select("id_pais").distinct().collect() if r.id_pais]
info_paises = {}

print(f"   📍 Consultando API para {len(paises_unicos)} país/es:")
for pid in paises_unicos:
    info = obtener_info_pais(pid)
    info_paises[pid] = info
    print(f"      ✅ ID {pid}: {info['pais_nombre']} - {info['moneda']}")

expr_nombre    = lit(None).cast("string")
expr_region    = lit(None).cast("string")
expr_capital   = lit(None).cast("string")
expr_poblacion = lit(0).cast("integer")
expr_moneda    = lit(None).cast("string")

for pid, info in info_paises.items():
    expr_nombre    = when(col("id_pais") == pid, lit(info["pais_nombre"])).otherwise(expr_nombre)
    expr_region    = when(col("id_pais") == pid, lit(info["region"])).otherwise(expr_region)
    expr_capital   = when(col("id_pais") == pid, lit(info["capital"])).otherwise(expr_capital)
    expr_poblacion = when(col("id_pais") == pid, lit(info["poblacion"])).otherwise(expr_poblacion)
    expr_moneda    = when(col("id_pais") == pid, lit(info["moneda"])).otherwise(expr_moneda)

df_clientes_final = df_clientes_t2 \
    .withColumn("pais_nombre",    expr_nombre) \
    .withColumn("pais_region",    expr_region) \
    .withColumn("pais_capital",   expr_capital) \
    .withColumn("pais_poblacion", expr_poblacion) \
    .withColumn("pais_moneda",    expr_moneda) \
    .withColumn("silver_timestamp", current_timestamp())

print(f"   ✅ Columnas API agregadas")
print(f"   📊 Registros finales: {df_clientes_final.count():,}")

print("\n💾 Guardando: shop_clientes (Delta)")
tabla_path = f"{SILVER_BASE}/shop_clientes"
df_clientes_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(tabla_path)
print(f"✅ Guardado en: {tabla_path}\n")


# =============================================================================
# 📦 DATASET 3/7: ARTICULOS
# =============================================================================

print("="*80)
print("📦 DATASET 3/7: ARTICULOS")
print("="*80)

df_articulos_bronze = spark.read.parquet(f"{BRONZE_PATH}/articulos")
print(f"📥 Registros en Bronze: {df_articulos_bronze.count():,}")

df_articulos_bronze_clean = df_articulos_bronze.toDF("raw_data", "_bronze_ingestion_timestamp", "_bronze_source_file", "_bronze_dataset")

print("\n🔧 TRANSFORMACIÓN 1: Parsing")
sc = split(col("raw_data"), ",")

df_articulos_t1 = df_articulos_bronze_clean \
    .withColumn("id",          sc.getItem(0).cast("integer")) \
    .withColumn("articulo",    trim(sc.getItem(1))) \
    .withColumn("descripcion", trim(sc.getItem(2))) \
    .withColumn("id_marca",    sc.getItem(3).cast("integer")) \
    .withColumn("estado",      sc.getItem(4).cast("integer")) \
    .withColumn("costo",       sc.getItem(25).cast("decimal(10,2)")) \
    .withColumn("sku",         trim(sc.getItem(30))) \
    .filter(col("id").isNotNull()).filter(col("id") != 0).dropDuplicates(["id"]) \
    .drop("raw_data", "_bronze_ingestion_timestamp", "_bronze_source_file", "_bronze_dataset")

print(f"   ✅ Parsing | 📊 Registros: {df_articulos_t1.count():,}")

print("\n🔧 TRANSFORMACIÓN 2: Categorización")

df_articulos_final = df_articulos_t1 \
    .withColumn("categoria_costo",
        when(col("costo") < 5, "ECONOMICO").when((col("costo") >= 5) & (col("costo") < 15), "MEDIO")
        .when(col("costo") >= 15, "PREMIUM").otherwise("SIN_DATOS")
    ) \
    .withColumn("costo_con_margen", round(col("costo") * 1.3, 2)) \
    .withColumn("estado_desc", when(col("estado") == 1, "ACTIVO").when(col("estado") == 0, "INACTIVO").otherwise("DESCONOCIDO")) \
    .withColumn("silver_timestamp", current_timestamp())

print(f"   ✅ Categorización | 📊 Registros: {df_articulos_final.count():,}")

print("\n💾 Guardando: articulos (Delta)")
df_articulos_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER_BASE}/articulos")
print(f"✅ Guardado\n")


# =============================================================================
# 📢 DATASET 4/7: CAMPANIAS_GP
# =============================================================================

print("="*80)
print("📢 DATASET 4/7: CAMPANIAS_GP")
print("="*80)

df_campanias_bronze = spark.read.parquet(f"{BRONZE_PATH}/campanias_gp")
print(f"📥 Registros en Bronze: {df_campanias_bronze.count():,}")

df_campanias_bronze_clean = df_campanias_bronze.toDF("raw_data", "_bronze_ingestion_timestamp", "_bronze_source_file", "_bronze_dataset")

print("\n🔧 TRANSFORMACIÓN 1: Parsing")
sc2 = split(col("raw_data"), ",")

df_campanias_t1 = df_campanias_bronze_clean \
    .withColumn("id", sc2.getItem(0).cast("integer")) \
    .withColumn("campania", trim(sc2.getItem(1))) \
    .withColumn("created", to_timestamp(regexp_replace(sc2.getItem(2), '"', ''), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("updated", to_timestamp(regexp_replace(sc2.getItem(3), '"', ''), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("estado", sc2.getItem(4).cast("integer")) \
    .filter(col("id").isNotNull()).filter(col("id") != 0).dropDuplicates(["id"]) \
    .drop("raw_data", "_bronze_ingestion_timestamp", "_bronze_source_file", "_bronze_dataset")

print(f"   ✅ Parsing | 📊 Registros: {df_campanias_t1.count():,}")

print("\n🔧 TRANSFORMACIÓN 2: Clasificación")

df_campanias_final = df_campanias_t1 \
    .withColumn("estado_desc", when(col("estado") == 1, "ACTIVA").when(col("estado") == 0, "INACTIVA").otherwise("DESCONOCIDO")) \
    .withColumn("anio_creacion", year(col("created"))) \
    .withColumn("mes_creacion", month(col("created"))) \
    .withColumn("silver_timestamp", current_timestamp())

print(f"   ✅ Clasificación | 📊 Registros: {df_campanias_final.count():,}")

print("\n💾 Guardando: campanias_gp (Delta)")
df_campanias_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER_BASE}/campanias_gp")
print(f"✅ Guardado\n")


# =============================================================================
# 📊 DATASET 5/7: SHOP_CARRITO_ESTADOS
# =============================================================================

print("="*80)
print("📊 DATASET 5/7: SHOP_CARRITO_ESTADOS")
print("="*80)

df_estados_bronze = spark.read.parquet(f"{BRONZE_PATH}/shop_carrito_estados")
print(f"📥 Registros en Bronze: {df_estados_bronze.count():,}")

print("\n🔧 TRANSFORMACIÓN 1: Normalización")

df_estados_t1 = df_estados_bronze.filter(col("id").isNotNull()).dropDuplicates(["id"]) \
    .withColumn("estado_descripcion", upper(trim(col("estado")))) \
    .withColumn("es_interno", col("interno").cast("integer"))

print(f"   ✅ Normalización | 📊 Registros: {df_estados_t1.count():,}")

print("\n🔧 TRANSFORMACIÓN 2: Categorización")

df_estados_final = df_estados_t1 \
    .withColumn("categoria_estado", when(col("es_interno") == 1, "INTERNO").when(col("es_interno") == 0, "EXTERNO").otherwise("NO_DEFINIDO")) \
    .withColumn("silver_timestamp", current_timestamp())

print(f"   ✅ Categorización | 📊 Registros: {df_estados_final.count():,}")

print("\n💾 Guardando: shop_carrito_estados (Delta)")
df_estados_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER_BASE}/shop_carrito_estados")
print(f"✅ Guardado\n")


# =============================================================================
# 💳 DATASET 6/7: SHOP_CARRITO_ESTADOS_PAGO
# =============================================================================

print("="*80)
print("💳 DATASET 6/7: SHOP_CARRITO_ESTADOS_PAGO")
print("="*80)

df_estados_pago_bronze = spark.read.parquet(f"{BRONZE_PATH}/shop_carrito_estados_pago")
print(f"📥 Registros en Bronze: {df_estados_pago_bronze.count():,}")

print("\n🔧 TRANSFORMACIÓN 1: Normalización")

df_estados_pago_t1 = df_estados_pago_bronze.filter(col("id").isNotNull()).dropDuplicates(["id"]) \
    .withColumn("estado_pago_desc", upper(trim(col("estado"))))

print(f"   ✅ Normalización | 📊 Registros: {df_estados_pago_t1.count():,}")

print("\n🔧 TRANSFORMACIÓN 2: Clasificación de riesgo")

df_estados_pago_final = df_estados_pago_t1 \
    .withColumn("nivel_riesgo",
        when(col("estado_pago_desc").isin(["APROBADO", "PAGADO", "ACREDITADO"]), "BAJO")
        .when(col("estado_pago_desc").isin(["PENDIENTE", "EN REVISION", "REVISION"]), "MEDIO")
        .when(col("estado_pago_desc").isin(["RECHAZADO", "CANCELADO", "ANULADO"]), "ALTO")
        .otherwise("SIN_CLASIFICAR")
    ) \
    .withColumn("silver_timestamp", current_timestamp())

print(f"   ✅ Clasificación | 📊 Registros: {df_estados_pago_final.count():,}")

print("\n💾 Guardando: shop_carrito_estados_pago (Delta)")
df_estados_pago_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER_BASE}/shop_carrito_estados_pago")
print(f"✅ Guardado\n")


# =============================================================================
# 💰 DATASET 7/7: SHOP_FORMAPAGO
# =============================================================================

print("="*80)
print("💰 DATASET 7/7: SHOP_FORMAPAGO")
print("="*80)

df_formapago_bronze = spark.read.parquet(f"{BRONZE_PATH}/shop_formapago")
print(f"📥 Registros en Bronze: {df_formapago_bronze.count():,}")

print("\n🔧 TRANSFORMACIÓN 1: Normalización")

cols_desc = [c for c in df_formapago_bronze.columns if c not in ["id", "_bronze_ingestion_timestamp", "_bronze_source_file", "_bronze_dataset"]]
col_forma = cols_desc[0] if cols_desc else "forma"

df_formapago_t1 = df_formapago_bronze.filter(col("id").isNotNull()).dropDuplicates(["id"]) \
    .withColumn("forma_pago_desc", upper(trim(col(col_forma))))

print(f"   ✅ Normalización | 📊 Registros: {df_formapago_t1.count():,}")

print("\n🔧 TRANSFORMACIÓN 2: Categorización")

df_formapago_final = df_formapago_t1 \
    .withColumn("tipo_medio",
        when(col("forma_pago_desc").like("%TARJETA%"), "TARJETA")
        .when(col("forma_pago_desc").like("%CREDITO%"), "TARJETA")
        .when(col("forma_pago_desc").like("%DEBITO%"), "DEBITO")
        .when(col("forma_pago_desc").like("%TRANSFERENCIA%"), "TRANSFERENCIA")
        .when(col("forma_pago_desc").like("%EFECTIVO%"), "EFECTIVO")
        .when(col("forma_pago_desc").like("%MERCADO%"), "BILLETERA_DIGITAL")
        .when(col("forma_pago_desc").like("%PAYPAL%"), "BILLETERA_DIGITAL")
        .otherwise("OTRO")
    ) \
    .withColumn("silver_timestamp", current_timestamp())

print(f"   ✅ Categorización | 📊 Registros: {df_formapago_final.count():,}")

print("\n💾 Guardando: shop_formapago (Delta)")
df_formapago_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER_BASE}/shop_formapago")
print(f"✅ Guardado\n")


# =============================================================================
# 📊 RESUMEN FINAL
# =============================================================================

print("\n" + "="*80)
print("📊 RESUMEN FINAL CAPA SILVER")
print("="*80)

tablas = [
    ("1", "shop_carrito",               "✅ ExchangeRate-API"),
    ("2", "shop_clientes",              "✅ REST Countries"),
    ("3", "articulos",                  "❌"),
    ("4", "campanias_gp",               "❌"),
    ("5", "shop_carrito_estados",       "❌"),
    ("6", "shop_carrito_estados_pago",  "❌"),
    ("7", "shop_formapago",             "❌"),
]

print(f"\n{'#':<3} {'Tabla Delta':<35} {'Registros':<12} {'API'}")
print("="*80)

total = 0
for num, tabla, api in tablas:
    try:
        df_check = spark.read.format("delta").load(f"{SILVER_BASE}/{tabla}")
        count = df_check.count()
        total += count
        print(f"{num:<3} {tabla:<35} {count:<12,} {api}")
    except Exception as e:
        print(f"{num:<3} {tabla:<35} {'ERROR':<12} ❌")

print("="*80)
print(f"{'':3} {'TOTAL':35} {total:<12,}")

print(f"""
✅ CAPA SILVER COMPLETADA
   • 7 archivos Bronze → 7 tablas Delta en lh_silver
   • Mínimo 2 transformaciones por tabla
   • 2 datasets con API (ExchangeRate + REST Countries)
   • Formato: Delta Lake

📁 Ubicación: {SILVER_BASE}

🔜 SIGUIENTE PASO: Notebook Gold → Crear tabla OBT
""")

StatementMeta(, ff06e322-6a34-421b-8572-048cbd47cc1a, 3, Submitted, Running, Running)

✅ Configuración Silver inicializada
📂 Origen Bronze: abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/e9dcb661-05e8-46bd-be1f-a9cb5ee27446/Files/bronze/shop_carrito
💎 Destino Silver: abfss://9a547f55-a8f7-4c89-b935-fb3da8226ed0@onelake.dfs.fabric.microsoft.com/4d897e0a-b68c-4840-9910-752862308e5b/Files/silver/shop_carrito
💾 Formato: Delta Lake

📋 7 datasets | Mínimo 2 transformaciones cada uno
🌐 2 datasets con enriquecimiento via API

🛒 DATASET 1/7: SHOP_CARRITO  (CON API)
📥 Registros en Bronze: 120,408

🔧 TRANSFORMACIÓN 1: Limpieza y normalización de tipos
   ✅ Duplicados eliminados | Fechas parseadas | Tipos casteados
   📊 Registros después de T1: 120,348

🔧 TRANSFORMACIÓN 2: Columnas derivadas y segmentación
   ✅ Columnas temporales + Segmentación aplicada
   📊 Registros después de T2: 120,348

🌐 TRANSFORMACIÓN 3 (CON API): Conversión de moneda
   ✅ API consultada: 2026-02-19
   💱 1 USD = 1452.25 ARS | 0.847 EUR | 5.22 BRL
   ✅ Columnas API agregadas
   